# Visium - SCA013

In [ ]:
# TODO: update Visium reader in harpy

## Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

In [ ]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _visium import visium

## Create sdata

In [ ]:
dataset_id = "SCA013"

path_outs = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium/{dataset_id}/COUNTS/{dataset_id}/{dataset_id}_reseq/outs'
counts_file = "raw_feature_bc_matrix.h5"
path_img = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium/{dataset_id}/IMAGES/{dataset_id}_HE_hires.tif'
output_path = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium/{dataset_id}/harpy/sdata.zarr'

pipeline_name = 'SpaceRanger'
species = 'mouse'
tissue_annotation_file = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium/{dataset_id}/tissue_annotation/tissue.geojson'

In [ ]:
# Create SpatialData from Space Ranger output
sdata = visium(
    path = path_outs, 
    counts_file = counts_file,
    dataset_id = dataset_id,
    fullres_image_file = path_img,
    # output = output_path
)
sdata

## Specify plotting parameters

In [ ]:
DPI = 200
MAX_DIM_FOR_PLOTS = 5000

HE_image = f'{dataset_id}_full_image'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

## Plot H&E

In [ ]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

## Plot Space Ranger tissue annotation

In [ ]:
# Check spots
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Space Ranger tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_labels(
    element=f"{dataset_id}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue', scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)


## Calculate pixel size

In [ ]:
# Determine pixel size in microns
import tifffile

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")


## Create bounding box for total capture area

In [ ]:
# Add bounding box for total capture area
union_geom = sdata.shapes[dataset_id].union_all()
bbox = union_geom.minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

In [ ]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=dataset_id, color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

In [ ]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

## Add tissue annotation

In [ ]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

In [ ]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Determine which spots are in tissue annotation
sdata = add_in_tissue(
    sdata, 
    table_layer = 'table', 
    tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
)

In [ ]:
# Check spots in tissue
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation spots"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_labels(
    element=f"{dataset_id}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue_annotation', table_layer='table', scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation_spots.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Total number of spots in table
n_spots = len(sdata.tables['table'].obs)
print(f"Number of spots in table: {n_spots}")

In [ ]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables['table'].obs[sdata.tables['table'].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

In [ ]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables['table'][sdata.tables['table'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

In [ ]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = sdata.tables['table'].obs['in_tissue_annotation'] == 1
umis_mask = np.array(sdata.tables['table'].X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

In [ ]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

## Capture effeciency metrics

In [ ]:
probe_set_df = pd.read_csv(f"{path_outs}/probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['table'][:, sdata.tables['table'].var['gene_ids'].isin(included)]

In [ ]:
# Total number of genes
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

In [ ]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum()}")

In [ ]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum()}")

In [ ]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum()}")

In [ ]:
# Total number of UMIs

import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


## Export unique UMI counts per gene

In [ ]:
adata_in_tissue = adata[adata.obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

## QC

In [ ]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
sdata = qc_metrics(
    sdata,
    'table',
    species
)

In [ ]:
# Plot QC metrics
for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:

    values = sdata.tables["table"].obs[qc_metric].to_numpy()
    vmin, vmax = np.percentile(values, [1, 99])
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
    
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = qc_metric

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, scale=best_scale, table_name='table', cmap="viridis", norm=norm
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=True,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'{qc_metric}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

In [ ]:
# for table_layer in [*sdata.tables]:
#     print(f'running: {table_layer}')

#     labels_layer=f"{dataset_id}_labels"
#     shapes_layer=dataset_id
    
#     plots_path_table = os.path.join(plots_path, table_layer)
#     os.makedirs(plots_path_table, exist_ok=True)
    
#     # Preprocess data
#     sdata = hp.tb.preprocess_transcriptomics(
#         sdata=sdata,
#         size_norm=False,
#         labels_layer=labels_layer,
#         table_layer=table_layer,
#         output_layer=f"{table_layer}_processed",
#         min_counts=10,
#         min_cells=5,
#         highly_variable_genes=True,
#         max_value_scale=10,
#         n_comps = 50,
#         update_shapes_layers=False,
#         overwrite=True
#     )
    
#     gc.collect()
    
#     # Leiden clustering
#     leiden_resolutions = [0.6]

#     for res in leiden_resolutions:
#         res_str = str(res).replace('.', '')
                
#         sdata = hp.tb.leiden(
#             sdata, 
#             labels_layer=labels_layer,
#             table_layer=f"{table_layer}_processed",
#             output_layer=f"{table_layer}_processed",
#             calculate_umap=True,
#             calculate_neighbors=True,
#             rank_genes=True,
#             n_neighbors=35,
#             n_pcs=30,
#             resolution=res,
#             key_added=f'leiden_{res_str}',
#             index_names_var=None,
#             index_positions_var=None,
#             random_state=100,
#             overwrite=True
#         )
        
#         hp.pl.cluster(
#             sdata, 
#             table_layer = f"{table_layer}_processed",
#             key_added=f'leiden_{res_str}',
#             output=os.path.join(plots_path_table, f'UMAP_leiden_{res_str}')
#         )
        
#         fig, ax = plt.subplots(figsize=figsize, dpi=300)
#         ax.set_xlabel("x", fontsize=12)
#         ax.set_ylabel("y", fontsize=12)

#         sdata.pl.render_images(
#             element=hne_img_layer
#         ).pl.render_labels(
#             element=labels_layer,
#             color=f'leiden_{res_str}',
#             table_name=f"{table_layer}_processed",
#             cmap="rainbow",
#             fill_alpha=1, 
#             contour_px=1, 
#             outline_alpha=0
#         ).pl.show(
#             coordinate_systems="global",
#             title=f"Leiden {res_str}",
#             figsize=figsize,
#             dpi=300,
#             colorbar=True,
#             ax=ax,
#         )

#         save_path = os.path.join(plots_path_lvl, f'leiden_{res_str}.png')
#         fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

#         plt.close(fig)

#         print("Plot saved at:", save_path)
    
#         gc.collect()